# 📄 Step 1 of RAG — Read, Chunk & Embed Documents

Before a RAG system can answer any question, it needs to **ingest** your documents. This notebook walks through the three stages of that pipeline:

```
Raw Text  ──►  Chunks  ──►  Embeddings
```

We'll use **Alice's Adventures in Wonderland** as our source document, experiment with several chunking strategies, and explore all three embedding types offered by the `BAAI/bge-m3` model.

> 💡 **Why does this matter?** The quality of your chunks and embeddings directly determines how well your RAG system can retrieve relevant context. Poor chunking = poor answers, no matter how good your LLM is.

## 0. Setup & Imports

In [195]:
import re
import textwrap
import numpy as np
from pathlib import Path

import munch
from munch import Munch
import yaml
from FlagEmbedding import BGEM3FlagModel
from sklearn.metrics.pairwise import cosine_similarity

### Load config

All tuneable parameters (chunk sizes, overlap, batch size, etc.) live in `config/config.yaml`. Centralising config means you only ever change values in one place.

In [196]:
# 📂 Load config
config_path = (Path.cwd() / "../config/config.yaml").resolve()

with open(config_path, "r") as f:
    config = Munch.fromYAML(f)

print("✅ Config loaded")
print(yaml.dump(config.toDict(), default_flow_style=False))

✅ Config loaded
chunking:
  book_file: Alice's Adventures in Wonderland by Lewis Carroll.txt
  fixed_size:
    chunk_size: 1000
    overlap: 200
  paragraph:
    min_chars: 80
  sentence:
    max_sentences: 8
    overlap_sentences: 2
  token:
    chunk_size: 400
    overlap: 50
embedding:
  batch_size: 16
  max_length: 512
  model: BAAI/bge-m3
groq:
  base_url: https://api.groq.com/openai/v1
  max_tokens: 100
  model: llama-3.1-8b-instant
  temperature: 0.7
supabase:
  database: postgres
  host: db.ewcugkjfmcxnwafzhqdf.supabase.co
  port: 5432
  user: postgres



---

## 1. Reading the Document

The first step is simply loading the raw text. In a real system this is where you'd also handle PDFs, Word docs, HTML pages, etc. For now we work with plain text.

In [197]:
# 📖 Load the source document
doc_path = Path("../assets/Alice's Adventures in Wonderland by Lewis Carroll.txt")

with open(doc_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"✅ Loaded document: {doc_path.name}")
print(f"   Total characters : {len(raw_text):,}")
print(f"   Total words      : {len(raw_text.split()):,}")
print(f"\n--- First 500 characters ---")
print(raw_text[:500])

✅ Loaded document: Alice's Adventures in Wonderland by Lewis Carroll.txt
   Total characters : 163,067
   Total words      : 29,432

--- First 500 characters ---
[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster 


### Light Cleaning

Raw files often contain artefacts (Project Gutenberg headers, `[Illustration]` tags, excessive whitespace). A quick clean-up before chunking avoids noise in our embeddings.

In [198]:
def clean_text(text: str) -> str:
    """Remove illustration tags and normalise whitespace."""
    # Drop [Illustration] markers
    text = re.sub(r"\[Illustration[^\]]*\]", "", text)
    # Collapse 3+ newlines into two (paragraph break)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Strip leading/trailing whitespace from each line
    text = "\n".join(line.strip() for line in text.splitlines())
    return text.strip()

clean = clean_text(raw_text)

print(f"Characters before cleaning : {len(raw_text):,}")
print(f"Characters after  cleaning : {len(clean):,}")
print(f"\n--- First 500 characters (cleaned) ---")
print(clean[:500])

Characters before cleaning : 163,067
Characters after  cleaning : 162,443

--- First 500 characters (cleaned) ---
Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

CHAPTER I.     Down the Rabbit-Hole
CHAPTER II.    The Pool of Tears
CHAPTER III.   A Caucus-Race and a Long Tale
CHAPTER IV.    The Rabbit Sends in a Little Bill
CHAPTER V.     Advice from a Caterpillar
CHAPTER VI.    Pig and Pepper
CHAPTER VII.   A Mad Tea-Party
CHAPTER VIII.  The Queen’s Croquet-Ground
CHAPTER IX.    The Mock Turtle’s Story
CHAPTER X.     The Lobster Quadrille
CHAPTER XI.    Who 


---

## 2. Chunking Strategies

> **Why chunk at all?**  
> Embedding models have a maximum context window (e.g., 512 tokens for `bge-m3`). A 150-page novel cannot be embedded as a single vector. More importantly, a focused, small chunk is *much* more likely to contain the exact answer than a huge block of text — and retrieval precision matters more than recall in RAG.

We'll compare four strategies:

| Strategy | Description |
|---|---|
| **Fixed-size** | Split every N characters, with optional overlap |
| **Sentence** | Split on sentence boundaries |
| **Paragraph** | Split on blank lines (natural document structure) |
| **Chapter (hierarchical)** | Split on chapter headings first, then sub-chunk |

---

### 2.1 Fixed-Size Chunking

The simplest strategy: slice the document every **N characters**, with an **overlap** so that context isn't lost at boundaries.

```
│◄── chunk_size ──►│
│         │◄── chunk_size ──►│
          │◄overlap►│
```

In [199]:
def fixed_size_chunks(text: str, chunk_size: int, overlap: int) -> list[str]:
    """
    Split text into overlapping fixed-size character chunks.

    Args:
        text       : The input text.
        chunk_size : Number of characters per chunk.
        overlap    : Number of characters shared between consecutive chunks.

    Returns:
        List of chunk strings.
    """
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        chunk = text[start : start + chunk_size]
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks


# Pull values from config
chunk_size = config.chunking.fixed_size.chunk_size
overlap    = config.chunking.fixed_size.overlap

fixed_chunks = fixed_size_chunks(clean, chunk_size=chunk_size, overlap=overlap)

print(f"Settings  : chunk_size={chunk_size}, overlap={overlap}")
print(f"Chunks    : {len(fixed_chunks)}")
print(f"Avg length: {sum(len(c) for c in fixed_chunks) / len(fixed_chunks):.0f} chars")
print(f"\n--- Example chunk [5] ---")
print(textwrap.fill(fixed_chunks[5], width=80))

Settings  : chunk_size=1000, overlap=200
Chunks    : 204
Avg length: 994 chars

--- Example chunk [5] ---
ough_ the earth! How funny it’ll seem to come out among the people that walk
with their heads downward! The Antipathies, I think—” (she was rather glad there
_was_ no one listening, this time, as it didn’t sound at all the right word)
“—but I shall have to ask them what the name of the country is, you know.
Please, Ma’am, is this New Zealand or Australia?” (and she tried to curtsey as
she spoke—fancy _curtseying_ as you’re falling through the air! Do you think you
could manage it?) “And what an ignorant little girl she’ll think me for asking!
No, it’ll never do to ask: perhaps I shall see it written up somewhere.”  Down,
down, down. There was nothing else to do, so Alice soon began talking again.
“Dinah’ll miss me very much to-night, I should think!” (Dinah was the cat.) “I
hope they’ll remember her saucer of milk at tea-time. Dinah my dear! I wish you
were down here with me! There

> 🔍 **Observation**: Fixed-size chunks are fast and predictable, but they often cut mid-sentence. The overlap mitigates this but doesn't eliminate it. Notice how the example chunk above may begin or end in the middle of a thought.

### 2.2 Sentence-Level Chunking

A step up in semantic coherence: we respect sentence boundaries, then group sentences together until we hit a target token budget.

In [200]:
def sentence_chunks(text: str, max_sentences: int, overlap_sentences: int) -> list[str]:
    """
    Split text by grouping consecutive sentences.

    Args:
        text              : The input text.
        max_sentences     : Maximum sentences per chunk.
        overlap_sentences : Number of sentences to carry over into the next chunk.

    Returns:
        List of chunk strings.
    """
    # Naïve sentence splitter — split on . ! ? followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    step = max_sentences - overlap_sentences
    for i in range(0, len(sentences), step):
        chunk = " ".join(sentences[i : i + max_sentences])
        if chunk:
            chunks.append(chunk)
    return chunks


max_sents  = config.chunking.sentence.max_sentences
overlap_s  = config.chunking.sentence.overlap_sentences

sent_chunks = sentence_chunks(clean, max_sentences=max_sents, overlap_sentences=overlap_s)

print(f"Settings  : max_sentences={max_sents}, overlap_sentences={overlap_s}")
print(f"Chunks    : {len(sent_chunks)}")
print(f"Avg length: {sum(len(c) for c in sent_chunks) / len(sent_chunks):.0f} chars")
print(f"\n--- Example chunk [5] ---")
print(textwrap.fill(sent_chunks[5], width=80))

Settings  : max_sentences=8, overlap_sentences=2
Chunks    : 182
Avg length: 1176 chars

--- Example chunk [5] ---
“I wonder if I shall fall right _through_ the earth! How funny it’ll seem to
come out among the people that walk with their heads downward! The Antipathies,
I think—” (she was rather glad there _was_ no one listening, this time, as it
didn’t sound at all the right word) “—but I shall have to ask them what the name
of the country is, you know. Please, Ma’am, is this New Zealand or Australia?”
(and she tried to curtsey as she spoke—fancy _curtseying_ as you’re falling
through the air! Do you think you could manage it?) “And what an ignorant little
girl she’ll think me for asking! No, it’ll never do to ask: perhaps I shall see
it written up somewhere.”  Down, down, down. There was nothing else to do, so
Alice soon began talking again. “Dinah’ll miss me very much to-night, I should
think!” (Dinah was the cat.) “I hope they’ll remember her saucer of milk at tea-
time.


> 🔍 **Observation**: Sentence chunks read more naturally and never cut mid-sentence. Each chunk is a coherent mini-paragraph.

### 2.3 Paragraph-Level Chunking

Paragraphs are the author's own unit of thought. Splitting on blank lines preserves semantic coherence without any heuristics.

In [201]:
def paragraph_chunks(text: str, min_chars: int) -> list[str]:
    """
    Split text on blank lines; discard very short paragraphs.

    Args:
        text      : The input text.
        min_chars : Minimum character count to keep a paragraph.

    Returns:
        List of chunk strings.
    """
    paragraphs = re.split(r"\n{2,}", text)
    return [p.strip() for p in paragraphs if len(p.strip()) >= min_chars]


min_chars = config.chunking.paragraph.min_chars

para_chunks = paragraph_chunks(clean, min_chars=min_chars)

print(f"Settings  : min_chars={min_chars}")
print(f"Chunks    : {len(para_chunks)}")
print(f"Avg length: {sum(len(c) for c in para_chunks) / len(para_chunks):.0f} chars")
print(f"Min length: {min(len(c) for c in para_chunks)} chars")
print(f"Max length: {max(len(c) for c in para_chunks)} chars")
print(f"\n--- Example chunk [10] ---")
print(textwrap.fill(para_chunks[10], width=80))

Settings  : min_chars=80
Chunks    : 587
Avg length: 250 chars
Min length: 80 chars
Max length: 1184 chars

--- Example chunk [10] ---
Down, down, down. There was nothing else to do, so Alice soon began talking
again. “Dinah’ll miss me very much to-night, I should think!” (Dinah was the
cat.) “I hope they’ll remember her saucer of milk at tea-time. Dinah my dear! I
wish you were down here with me! There are no mice in the air, I’m afraid, but
you might catch a bat, and that’s very like a mouse, you know. But do cats eat
bats, I wonder?” And here Alice began to get rather sleepy, and went on saying
to herself, in a dreamy sort of way, “Do cats eat bats? Do cats eat bats?” and
sometimes, “Do bats eat cats?” for, you see, as she couldn’t answer either
question, it didn’t much matter which way she put it. She felt that she was
dozing off, and had just begun to dream that she was walking hand in hand with
Dinah, and saying to her very earnestly, “Now, Dinah, tell me the truth: did you
ever 

> 🔍 **Observation**: Paragraph chunks are variable in length — some paragraphs are one line of dialogue, others are long descriptive passages. The high max-length here signals that we may want to sub-chunk very large paragraphs before embedding.

### 2.4 Hierarchical (Chapter-then-Paragraph) Chunking

The most structured approach: first split on chapter headings to capture *where* in the document a chunk lives, then sub-chunk each chapter. This lets you attach rich metadata (chapter title) to every chunk.

```
CHAPTER I  ──► paragraph 1, paragraph 2, …
CHAPTER II ──► paragraph 1, paragraph 2, …
…
```

In [202]:
def hierarchical_chunks(text: str, min_chars: int) -> list[dict]:
    """
    Split by chapter, then by paragraph within each chapter.

    Returns a list of dicts with keys:
        - 'chapter' : chapter heading string
        - 'text'    : the paragraph text
    """
    # Match CHAPTER headings (e.g. "CHAPTER I.\nDown the Rabbit-Hole")
    chapter_pattern = re.compile(
        r"(CHAPTER [IVXLC]+\.\s*\n[^\n]+)", re.MULTILINE
    )
    splits = chapter_pattern.split(text)

    results = []
    current_chapter = "Preamble"

    for segment in splits:
        segment = segment.strip()
        if not segment:
            continue
        if chapter_pattern.match(segment):
            current_chapter = " — ".join(segment.splitlines()[:2])
        else:
            # Sub-chunk this chapter body by paragraph
            paragraphs = re.split(r"\n{2,}", segment)
            for para in paragraphs:
                para = para.strip()
                if len(para) >= min_chars:
                    results.append({"chapter": current_chapter, "text": para})

    return results


hier_chunks = hierarchical_chunks(clean, min_chars=min_chars)

print(f"Total chunks : {len(hier_chunks)}")
print(f"Unique chapters found:")
for ch in dict.fromkeys(c["chapter"] for c in hier_chunks):
    count = sum(1 for c in hier_chunks if c["chapter"] == ch)
    print(f"  {ch!r:55s}  → {count} chunks")

print(f"\n--- Example hierarchical chunk ---")
example = hier_chunks[20]
print(f"Chapter : {example['chapter']}")
print(f"Text    : {textwrap.fill(example['text'][:300], width=80)}")

Total chunks : 587
Unique chapters found:
  'Preamble'                                               → 1 chunks
  'CHAPTER I. — Down the Rabbit-Hole'                      → 22 chunks
  'CHAPTER II. — The Pool of Tears'                        → 24 chunks
  'CHAPTER III. — A Caucus-Race and a Long Tale'           → 37 chunks
  'CHAPTER IV. — The Rabbit Sends in a Little Bill'        → 37 chunks
  'CHAPTER V. — Advice from a Caterpillar'                 → 51 chunks
  'CHAPTER VI. — Pig and Pepper'                           → 50 chunks
  'CHAPTER VII. — A Mad Tea-Party'                         → 64 chunks
  'CHAPTER VIII. — The Queen’s Croquet-Ground'             → 54 chunks
  'CHAPTER IX. — The Mock Turtle’s Story'                  → 60 chunks
  'CHAPTER X. — The Lobster Quadrille'                     → 52 chunks
  'CHAPTER XI. — Who Stole the Tarts?'                     → 44 chunks
  'CHAPTER XII. — Alice’s Evidence'                        → 91 chunks

--- Example hierarchical chunk ---


> 🔍 **Observation**: Each chunk now carries its chapter label. In a real RAG system you'd store this as metadata in your vector database, enabling pre-filtering like *'search only within Chapter V'*.

### 2.5 Comparing Chunking Strategies

Let's put the numbers side by side so we can reason about the trade-offs.

In [203]:
strategies = {
    "Fixed-size": fixed_chunks,
    "Sentence": sent_chunks,
    "Paragraph": para_chunks,
    "Hierarchical": [c["text"] for c in hier_chunks],
}

print(f"{'Strategy':<16} {'Count':>6} {'Avg chars':>10} {'Min':>7} {'Max':>7}")
print("-" * 52)
for name, chunks in strategies.items():
    lengths = [len(c) for c in chunks]
    print(
        f"{name:<16} {len(chunks):>6} {sum(lengths)/len(lengths):>10.0f} "
        f"{min(lengths):>7} {max(lengths):>7}"
    )

Strategy          Count  Avg chars     Min     Max
----------------------------------------------------
Fixed-size          204        994      42    1000
Sentence            182       1176     228    3678
Paragraph           587        250      80    1184
Hierarchical        587        250      80    1184


---

## 3. Embeddings with `BAAI/bge-m3`

> **What is an embedding?**  
> An embedding maps a piece of text to a point in high-dimensional space so that semantically similar texts land close together. *"dog"* and *"puppy"* should have similar embeddings; *"dog"* and *"mortgage"* should not.

`BAAI/bge-m3` is unusual because it produces **three kinds** of embeddings from a single model pass:

| Type | Also called | Shape | Best for |
|---|---|---|---|
| **Dense** | Semantic vector | `(1, 1024)` | Capturing meaning and synonyms |
| **Sparse** | Lexical weights | `dict{token_id: weight}` | Exact keyword matching (like BM25) |
| **ColBERT** | Multi-vector | `(n_tokens, 1024)` | Fine-grained, token-level matching |

Combining all three is called **hybrid retrieval** — the backbone of production RAG systems.

### 3.1 Load the Model

In [204]:
print("Loading BAAI/bge-m3 (will download on first run — ~2 GB)...")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("✅ Model ready")

Loading BAAI/bge-m3 (will download on first run — ~2 GB)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 50484.00it/s]


✅ Model ready


### 3.2 Embed a Small Batch — Inspecting All Three Types

Before embedding the whole book, let's embed just a handful of chunks and inspect the outputs so we can build intuition.

In [205]:
# Use a small, readable sample from paragraph chunks
sample_docs = para_chunks[5:10]

print("Sample documents:")
for i, doc in enumerate(sample_docs):
    print(f"  [{i}] {doc[:80]}…")

sample_embeddings = model.encode(
    sample_docs,
    batch_size=config.embedding.batch_size,
    max_length=config.embedding.max_length,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
)

print(f"\nDense embeddings shape    : {sample_embeddings['dense_vecs'].shape}")
print(f"Sparse (first doc entries): {len(sample_embeddings['lexical_weights'][0])} non-zero tokens")
print(f"ColBERT (first doc) shape : {sample_embeddings['colbert_vecs'][0].shape}")

Sample documents:
  [0] The rabbit-hole went straight on like a tunnel for some way, and then
dipped sud…
  [1] Either the well was very deep, or she fell very slowly, for she had
plenty of ti…
  [2] “Well!” thought Alice to herself, “after such a fall as this, I shall
think noth…
  [3] Down, down, down. Would the fall _never_ come to an end? “I wonder how
many mile…
  [4] Presently she began again. “I wonder if I shall fall right _through_
the earth! …

Dense embeddings shape    : (5, 1024)
Sparse (first doc entries): 33 non-zero tokens
ColBERT (first doc) shape : (49, 1024)


#### Dense vector — a snapshot

A dense vector is a single fixed-length array. Every element encodes some latent feature of the text.

In [206]:
dense_vec = sample_embeddings["dense_vecs"][0]

print(f"Vector dimensionality : {dense_vec.shape[0]}")
print(f"First 10 values       : {dense_vec[:10].round(4)}")
print(f"L2 norm               : {np.linalg.norm(dense_vec):.4f}  (should be ≈ 1 — it's normalised)")

Vector dimensionality : 1024
First 10 values       : [ 0.016   0.044  -0.014   0.0681 -0.0262 -0.0199  0.0277  0.0294 -0.0194
  0.0087]
L2 norm               : 0.9995  (should be ≈ 1 — it's normalised)


#### Sparse vector — lexical fingerprint

A sparse vector is a dictionary of `{token_id: importance_weight}`. Tokens absent from the text simply aren't in the dict — efficient and interpretable.

In [207]:
print(f"For this chunk: \n\n{sample_docs[0]}\n")

sparse_vec = sample_embeddings["lexical_weights"][0]

# Sort by weight descending
top_tokens = sorted(sparse_vec.items(), key=lambda x: x[1], reverse=True)[:10]

print(f"Non-zero token entries : {len(sparse_vec)}")
print(f"\nTop-10 tokens by weight:")
for token_id, weight in top_tokens:
    # Decode the token using the model's tokeniser
    token_str = model.tokenizer.decode([int(token_id)])
    print(f"  Token {token_id:>6}  ({token_str!r:>12})  weight={weight:.4f}")

For this chunk: 

The rabbit-hole went straight on like a tunnel for some way, and then
dipped suddenly down, so suddenly that Alice had not a moment to think
about stopping herself before she found herself falling down a very
deep well.

Non-zero token entries : 33

Top-10 tokens by weight:
  Token  66083  (     'Alice')  weight=0.2590
  Token   5299  (      'well')  weight=0.2205
  Token  70919  (      'hole')  weight=0.2130
  Token 152131  (     'rabbi')  weight=0.1772
  Token  80208  (    'tunnel')  weight=0.1667
  Token  80560  (  'straight')  weight=0.1440
  Token     98  (        'on')  weight=0.1417
  Token   6817  (      'fall')  weight=0.1399
  Token  53894  (      'deep')  weight=0.1292
  Token   7565  (      'down')  weight=0.1164


> 🔍 **Observation**: The highest-weighted tokens are exactly the words that characterise this passage. This is what makes sparse embeddings excellent at keyword retrieval — they're essentially a learned, weighted version of BM25.

#### ColBERT — multi-vector representation

ColBERT produces **one vector per token**, not one per document. At retrieval time, similarity is computed using **MaxSim**: for each query token, find the *most similar* document token, then average those scores.

In [208]:
colbert_vecs = sample_embeddings["colbert_vecs"][0]

print(f"ColBERT matrix shape : {colbert_vecs.shape}")
print(f"  → {colbert_vecs.shape[0]} tokens, each with a {colbert_vecs.shape[1]}-dim vector")
print(f"\nFirst token vector (first 8 dims): {colbert_vecs[0, :8].round(4)}")

ColBERT matrix shape : (49, 1024)
  → 49 tokens, each with a 1024-dim vector

First token vector (first 8 dims): [-0.0117 -0.0273 -0.0023 -0.0581  0.0018  0.0334 -0.0208 -0.0604]


---

## 4. Retrieval — Comparing Embedding Types

Now let's embed all paragraph chunks and run a query using each of the three methods. This mirrors what a real vector database does at search time.

### 4.1 Embed All Paragraph Chunks

In [209]:
# We'll work with paragraph chunks — a good balance of coherence and size
docs_to_embed = para_chunks

print(f"Embedding {len(docs_to_embed)} paragraph chunks…")

doc_embeddings = model.encode(
    docs_to_embed,
    batch_size=config.embedding.batch_size,
    max_length=config.embedding.max_length,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
)

dense_doc_vecs    = doc_embeddings["dense_vecs"]
sparse_doc_vecs   = doc_embeddings["lexical_weights"]
colbert_doc_vecs  = doc_embeddings["colbert_vecs"]

print(f"✅ Done — dense matrix shape: {dense_doc_vecs.shape}")

Embedding 587 paragraph chunks…


Inference Embeddings: 100%|██████████| 37/37 [00:10<00:00,  3.58it/s]


✅ Done — dense matrix shape: (587, 1024)


### 4.2 Define the Query & Embed It

In [210]:
query = "What did Alice find at the bottom of the rabbit hole?"

print(f"Query: {query}")

q_embeddings = model.encode(
    [query],
    batch_size=1,
    max_length=config.embedding.max_length,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
)

q_dense   = q_embeddings["dense_vecs"]
q_sparse  = q_embeddings["lexical_weights"][0]
q_colbert = q_embeddings["colbert_vecs"][0]

Query: What did Alice find at the bottom of the rabbit hole?


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 43.82it/s]


### 4.3 Dense Retrieval — Cosine Similarity

The classic semantic search: embed both query and documents, then rank by cosine similarity.

In [211]:
dense_scores = cosine_similarity(q_dense, dense_doc_vecs)[0]
dense_ranked = np.argsort(dense_scores)[::-1]

print(f"{'='*70}")
print(f"🔵 Dense Retrieval Results  (query: '{query}')")
print(f"{'='*70}")
for rank, idx in enumerate(dense_ranked[:3], 1):
    print(f"\nRank {rank} | Score: {dense_scores[idx]:.4f}")
    print(textwrap.fill(docs_to_embed[idx][:300], width=70))

🔵 Dense Retrieval Results  (query: 'What did Alice find at the bottom of the rabbit hole?')

Rank 1 | Score: 0.6478
The rabbit-hole went straight on like a tunnel for some way, and then
dipped suddenly down, so suddenly that Alice had not a moment to think
about stopping herself before she found herself falling down a very
deep well.

Rank 2 | Score: 0.6188
Alice did not quite know what to say to this: so she helped herself to
some tea and bread-and-butter, and then turned to the Dormouse, and
repeated her question. “Why did they live at the bottom of a well?”

Rank 3 | Score: 0.6168
“_That_ you won’t!” thought Alice, and, after waiting till she fancied
she heard the Rabbit just under the window, she suddenly spread out
her hand, and made a snatch in the air. She did not get hold of
anything, but she heard a little shriek and a fall, and a crash of
broken glass, from which she c


### 4.4 Sparse Retrieval — Lexical Overlap

Instead of cosine similarity over a dense vector, sparse retrieval computes a **dot product over shared vocabulary tokens** — essentially a learned TF-IDF.

In [212]:
def sparse_score(query_sparse: dict, doc_sparse: dict) -> float:
    """Dot product over shared token IDs."""
    shared = set(query_sparse.keys()) & set(doc_sparse.keys())
    return sum(query_sparse[t] * doc_sparse[t] for t in shared)


sparse_scores = [sparse_score(q_sparse, d) for d in sparse_doc_vecs]
sparse_ranked = np.argsort(sparse_scores)[::-1]

print(f"{'='*70}")
print(f"🟡 Sparse Retrieval Results  (query: '{query}')")
print(f"{'='*70}")
for rank, idx in enumerate(sparse_ranked[:3], 1):
    print(f"\nRank {rank} | Score: {sparse_scores[idx]:.4f}")
    print(textwrap.fill(docs_to_embed[idx][:300], width=70))

🟡 Sparse Retrieval Results  (query: 'What did Alice find at the bottom of the rabbit hole?')

Rank 1 | Score: 0.1411
“You couldn’t have wanted it much,” said Alice; “living at the bottom
of the sea.”

Rank 2 | Score: 0.1304
Alice tried to fancy to herself what such an extraordinary way of
living would be like, but it puzzled her too much, so she went on:
“But why did they live at the bottom of a well?”

Rank 3 | Score: 0.1225
“It was much pleasanter at home,” thought poor Alice, “when one wasn’t
always growing larger and smaller, and being ordered about by mice and
rabbits. I almost wish I hadn’t gone down that rabbit-hole—and yet—and
yet—it’s rather curious, you know, this sort of life! I do wonder what
_can_ have happe


### 4.5 ColBERT Retrieval — MaxSim

MaxSim asks: *for each query token, what is the most similar document token?* It then averages those per-token maximums. This is more compute-intensive but captures nuanced, fine-grained similarity.

In [213]:
def colbert_maxsim(query_vecs: np.ndarray, doc_vecs: np.ndarray) -> float:
    """MaxSim: mean over query tokens of their max similarity to any doc token."""
    sims = np.dot(query_vecs, doc_vecs.T)   # (n_query_tokens, n_doc_tokens)
    return float(np.mean(np.max(sims, axis=1)))


colbert_scores = [colbert_maxsim(q_colbert, d) for d in colbert_doc_vecs]
colbert_ranked = np.argsort(colbert_scores)[::-1]

print(f"{'='*70}")
print(f"🟢 ColBERT Retrieval Results  (query: '{query}')")
print(f"{'='*70}")
for rank, idx in enumerate(colbert_ranked[:3], 1):
    print(f"\nRank {rank} | Score: {colbert_scores[idx]:.4f}")
    print(textwrap.fill(docs_to_embed[idx][:300], width=70))

🟢 ColBERT Retrieval Results  (query: 'What did Alice find at the bottom of the rabbit hole?')

Rank 1 | Score: 0.6721
The rabbit-hole went straight on like a tunnel for some way, and then
dipped suddenly down, so suddenly that Alice had not a moment to think
about stopping herself before she found herself falling down a very
deep well.

Rank 2 | Score: 0.6379
CHAPTER I.     Down the Rabbit-Hole CHAPTER II.    The Pool of Tears
CHAPTER III.   A Caucus-Race and a Long Tale CHAPTER IV.    The Rabbit
Sends in a Little Bill CHAPTER V.     Advice from a Caterpillar
CHAPTER VI.    Pig and Pepper CHAPTER VII.   A Mad Tea-Party CHAPTER
VIII.  The Queen’s Croquet-

Rank 3 | Score: 0.6286
“It was much pleasanter at home,” thought poor Alice, “when one wasn’t
always growing larger and smaller, and being ordered about by mice and
rabbits. I almost wish I hadn’t gone down that rabbit-hole—and yet—and
yet—it’s rather curious, you know, this sort of life! I do wonder what
_can_ have happe


### 4.6 Side-by-Side Comparison

Which chunk ends up at rank 1 under each method?

In [214]:
print(f"Query: '{query}'\n")
print(f"{'Method':<12} {'Score':>8}   Top result (first 120 chars)")
print("-" * 70)

for label, scores, ranked, color in [
    ("Dense",    dense_scores,   dense_ranked,   "🔵"),
    ("Sparse",   sparse_scores,  sparse_ranked,  "🟡"),
    ("ColBERT",  colbert_scores, colbert_ranked, "🟢"),
]:
    top_idx = ranked[0]
    snippet = docs_to_embed[top_idx][:120].replace("\n", " ")
    print(f"{color} {label:<10} {scores[top_idx]:>8.4f}   {snippet}…")

Query: 'What did Alice find at the bottom of the rabbit hole?'

Method          Score   Top result (first 120 chars)
----------------------------------------------------------------------
🔵 Dense        0.6478   The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had n…
🟡 Sparse       0.1411   “You couldn’t have wanted it much,” said Alice; “living at the bottom of the sea.”…
🟢 ColBERT      0.6721   The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had n…


---

## 5. Putting It All Together — A Minimal Ingestion Pipeline

In a real RAG system, you'd run something like the function below once at setup time and store the results in a vector database (e.g., Supabase pgvector). For now, we hold everything in memory.

> This is the function you'd call in your actual RAG pipeline — it reads the document, chunks it, and embeds every chunk in one go.

In [215]:
def ingest_document(
    text: str,
    model: BGEM3FlagModel,
    chunk_size: int,
    overlap: int,
    batch_size: int,
    max_length: int,
) -> dict:
    """
    Full ingestion pipeline: clean → chunk → embed.

    Returns a dict with:
        - 'chunks'  : list of chunk strings
        - 'dense'   : np.ndarray of dense vectors
        - 'sparse'  : list of sparse dicts
        - 'colbert' : list of ColBERT matrices
    """
    # 1. Clean
    cleaned = clean_text(text)

    # 2. Chunk  (using fixed-size as the default — easy to swap out)
    chunks = fixed_size_chunks(cleaned, chunk_size=chunk_size, overlap=overlap)
    print(f"  📄 {len(chunks)} chunks created")

    # 3. Embed
    embeddings = model.encode(
        chunks,
        batch_size=batch_size,
        max_length=max_length,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=True,
    )
    print(f"  🧮 Embeddings computed — dense shape: {embeddings['dense_vecs'].shape}")

    return {
        "chunks": chunks,
        "dense":  embeddings["dense_vecs"],
        "sparse": embeddings["lexical_weights"],
        "colbert": embeddings["colbert_vecs"],
    }


print("Running full ingestion pipeline…")
index = ingest_document(
    text=raw_text,
    model=model,
    chunk_size=config.chunking.fixed_size.chunk_size,
    overlap=config.chunking.fixed_size.overlap,
    batch_size=config.embedding.batch_size,
    max_length=config.embedding.max_length,
)
print("✅ Ingestion complete — ready for retrieval!")

Running full ingestion pipeline…
  📄 204 chunks created


Inference Embeddings: 100%|██████████| 13/13 [00:12<00:00,  1.01it/s]

  🧮 Embeddings computed — dense shape: (204, 1024)
✅ Ingestion complete — ready for retrieval!


---

## 6. Summary & Key Takeaways

| Topic | What we learned |
|---|---|
| **Chunking** | Documents must be split before embedding — model context windows are finite |
| **Fixed-size** | Fast, simple, but cuts across sentence boundaries |
| **Sentence / Paragraph** | More semantically coherent; variable length |
| **Hierarchical** | Preserves document structure; enables metadata-based pre-filtering |
| **Dense embeddings** | Single vector per chunk; great for semantic similarity |
| **Sparse embeddings** | Token-weight dict; great for keyword/exact-match retrieval |
| **ColBERT embeddings** | Per-token vectors + MaxSim; highest precision, highest cost |
| **Hybrid retrieval** | Combining all three typically outperforms any single method |

### What comes next?

In the **RAG exercise** you'll take these embeddings and:
1. Store them in **Supabase pgvector**
2. Query the database with a user question
3. Pass the retrieved chunks as context to **Groq / LLaMA** to generate an answer

> 💡 **Rule of thumb**: Start with paragraph chunking + dense embeddings. Switch to hybrid retrieval only when you have evidence that the simpler approach isn't good enough.